day02

1. 웹크롤링
 : 웹페이지에서 필요한 데이터를 수집하는 것

1) requests
- 웹 사이트의 요청을 처리하는 라이브러리
- HTML문서를 가져올 때 사용

2) BeatifulSoup
 : HTML 구문을 해석해서 필요한 내용만 추출하는 패키지

3) 헤더(header)

(1) User-Agent : 사용자 소프트웨어 식별정보
- 내가 직접 접속하는 것과 requests 통해 접속하는
  user-agent가 다르기 때문에 크롤링이 안될 수 있다
  (406 코드)
- 필요성 :
	무분별한 크롤링 서버 과부화를 막기 위해
	프로그램을 통하여 접속하는 것을 차단하는 사이트
	이때 헤더를 사용
2) 내 User-Agent 확인 사이트

2. 워드 클라우드(word cloud)
- 시각화 기술 중 하나
- 수집한 단어들을 기반으로 단어마다 가중치를 부여하고 중요도 표현
- 중요도 높은 단어는 굵고 가운데로 표현
- matplotlib과 같이 사용

3. 데이터 전처리
 : 원본 데이터를 분석 목적에 맞게 다듬는 작업
- 머신러닝 모델은 결국 데이터에서 규칙을 배우는 것
- 데이터가 엉망이면, 엉망인 결과가 나오게 된다
- 깨끗한 데이터가 결과를 크게 좌우하는 경우가 많다
- 실무에서는 전체 작업 시간의 절반 이상을 전처리에 쓴다

+) 전처리 종류
- 결측치 처리
- 이상치 처리
- 자료형 변환
- 스케일링
...

4. 결측치 처리

1) 결측지
 : 아무 값도 없이 비어 있는 것(누락 데이터)
- 판다스는 결측치를 nan(Not a Number)라는 값으로 표현
- 머신러닝 모델은 nan이 들어오면 계산을 못하는 경우가 많다
- 그래서 분석을 시작하기 전 결측치를 어떻게 처리할 지 정해야 한다

2) 결측치 확인
- df.isnull().sum() : 열마다 결측치 개수 확인
- df.isnull().sum().sum() : 전체 데이터에서 결측치 개수 확인
- df["열"].isnull().sum() : 특정 컬럼의 결측치 개수 확인

3) 결측치 처리
(1) 결측치 처리 : dropna()
- 가장 단순한 처리법
- 결측치가 포함된 열 또는 행을 삭제
- df.dropna() : NaN이 하나라도 있는 행을 전부 삭제
- 옵션
	axis = 0(행 삭제, 기본값), 1(열 삭제)
	how = "any"(하나라도 NaN, 기본값), "all"(모두 NaN)
	subset = 특정 열에 NaN이 있는 행만 골라서 삭제
	thresh = 값이 최소 몇개 이상 있는 행은 남김

(2) 결측치 채우기 - fillna()
- 삭제하는 대신 결측치를 적당한 값으로 채우는 방법
- 데이터를 버리지 않아도 되는 것이 장점이다
- df.fillna(값) : 모든 NaN을 지정한 값으로 채운다

메소드
=======================================================
sr.fillna(0)	NaN을 0으로 채운다
df.fillna({열:값,...}) 열마다 다른 값으로 채움
sr.fillna(sr.mean()) 그 열의 평균으로 채움
sr.fillna(sr.mode()[0]) 그 열의 최빈값으로 채움
sr.ffill() : 바로 위의 값으로 채움
sr.bfill() : 바로 아래 칸의 값으로 채움

(3) 평균 vs 중앙값 vs 최빈값으로 채우기
채울 값			사용하는 경우
=========================================================
평균(mean)		숫자 데이터, 값이 고르게 퍼져 있을 때
중앙값(median)		숫자 데이터, 튀는 값(이상치)이 있을 때
최빈값(mode)		범주형(문자, 등급) 데이터

In [ ]:
## 멜론 차트 순위 웹 크롤링
> 1 ~ 50위 순위, 곡명 가수
import requests
from bs4 import BeautifulSoup as bs

url = "https://www.melon.com/chart/index.htm"
# 헤더 설정
header = {"User-Agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36"} 
response = requests.get(url, headers=header)
print(response.status_code) # 4xx : 사용자 잘못(잘못된 요청)
soup = bs(response.text, "html.parser")
# print(soup)
# 멜론 차트 1 ~ 50위 크롤링

# 1위 가져오기
rank = soup.find_all("span", class_="rank")[1].text
print(rank)

# 1위 ~ 50위
# 1) 순위, 곡명, 가수명이 들어간 행 가져오기
songs = soup.find_all("tr", class_="lst50")
# print(songs)

# 멜론차트 순위를 저장할 리스트
melon_li = []

# 2) 행마다 순위, 곡명, 가수명 추출
for song in songs :
    rank = song.find("span", class_="rank").text + "위"
    # print(rank)
    # 곡명 추출
    title = song.find_all("a")[2].text
    # print(title)
    # 가수명 추출
    singer = song.find_all("a")[3].text
    # print(singer)
    # print(f"{rank} : {title}, 가수는 {singer}")
    melon_li.append([rank, title, singer])
    print(melon_li)
import pandas as pd
# 가져온 순위를 데이터 프레임으로 생성
melon_df = pd.DataFrame(melon_li, columns=["순위", "노래제목", "가수"])

melon_df = melon_df.set_index("순위") # 순위를 인덱스로 설정
# 판다스 csv로 저장
melon_df.to_csv("./memo_rank.csv", encoding="utf-8-sig")

melon_df
## 워드 클라우드
from wordcloud import WordCloud
import matplotlib.pyplot as plt

di = {"Apple":12, "Banana":5, "Melon":8, "Grape":2}
wc = wordcloud.WordCloud(font_path="C:/Windows/Fonts/malgun.ttf")
# 한글 깨짐 방지
cloud = wc.generate_from_frequencies(di)
plt.imshow(cloud) # 워드 클라우드 출력
plt.axis("off") # 축 비활성화
plt.show()
melon_df
# <실습>
# 1위 ~ 50위에서 가수명으로 워드 클라우드 만들기

# 가수 출현 빈도수
print(melon_df["가수"].value_counts())
# 워드 클라우드 설정
wc = wordcloud.WordCloud(font_path="C:/Windows/Fonts/malgun.ttf",
                        background_color="ivory", # 배경색
                        max_words=20) # 최대 개수
cloud = wc.generate_from_frequencies(melon_df["가수"].value_counts())
plt.imshow(cloud)
plt.axis("off")
plt.show()
## 결측치 처리
import pandas as pd
import seaborn as sns

# 타이타닉 데이터 불러오기
titanic = sns.load_dataset("titanic")

sub = titanic[["age", "embarked", "deck"]]

sub.head()
sub.info()
# Non-null Count를 통해 결측치 확인 가능
# 칸 별로 비어있는지 True/False로 확인
print(sub.isnull())
# 열마다 결측치 개수 확인(가장 많이 사용하는 방법)
print(sub.isnull().sum())
# 데이터 전체 결측치 개수 확인
print(f"전체 결측치 개수 : {sub.isnull().sum().sum()}")
print(f"나이열 결측치 개수 : {sub['age'].isnull().sum()}")
## 결측치 처리
## NaN이 하나라도 있는 행을 모두 삭제
clean = sub.dropna()
print(f"남은 행 수 : {len(clean)} / 원래 : {len(sub)}")
# 891개 중 182개만 남음 => deck이 많이 비어서
# 다 지우니 데이터가 확 줄어들었다
# => 결측치 삭제의 위험성이다(중요한 데이터가 삭제될 위험이 있음)
# 'age'열이 빈 행만 삭제
kept = sub.dropna(subset=["age"])
print(f"남은 행 수 : {len(kept)} / 원래 : {len(sub)}")
# 필요한 열의 결측치만 지우니 데이터를 훨씬 아낄 수 있다
# 열 방향(axis=1)으로, NaN이 하나라도 있는 "열" 삭제
print(titanic.columns.tolist()) # 전체 컬럼명
print(titanic.isnull().sum())
print("="*10)
print(titanic.dropna(axis=1).columns.tolist())
# NaN이 하나라도 있는 열은 삭제되어 있음
# 삭제는 편하지만 위험하다
# - 결측치가 아주 적을 때나
# - 그 열/행이 어차피 못 쓸 때
# 이 경우일 경우 삭제한다
## 결측치 채우기
print(titanic["age"].head(10)) # 6번째 데이터가 NaN
# 1) NaN을 0으로 채우기
print(titanic['age'].fillna(0).head(10))
# 5번행의 나이가 0이 되었다
# 0으로 처리하면, 평균이 확 낮아지는 등 데이터가 왜곡된다
# 그래서 숫자는 보통 평균이나 중앙값으로 채운다
# 2) age열의 결측값을 평균으로 채우기
print(f"age의 평균 : {titanic['age'].mean() : .2f}")
print(titanic['age'].fillna(titanic['age'].mean()).head(10))
# 3) 위 칸 데이터 이어받기
print(titanic["age"].ffill().head(10))
# 바로 위 행(4번)의 값을 그대로 이어받음
# 4) 밑의 데이터 가져오기
print(titanic['age'].bfill().head(10))
# 바로 아래 행(6번)의 값을 가져옴
# 문자열은 평균을 계산할 수 없다
# => 이런 경우, 가장 많이 나온 값(최빈값)으로 채운다

deck_mode = titanic['deck'].mode()[0] # deck열의 최빈값 => c
print(f"deck열의 최빈값 : {deck_mode}")
print(titanic['deck'].head(10))
print("==== 최빈값으로 채운 후 ====")
print(titanic['deck'].fillna(deck_mode).head(10))
# NaN이던 칸들이 C로 채워진다
# <결측치 처리 실습>
# 1) 열마다 결측치 개수 확인
print(titanic.isnull().sum())
# 2) deck열은 결측치 개수가 너무 많아 열을 삭제
titanic = titanic.drop(columns="deck")
# 3) age열의 결측치는 중앙값으로 채우기
titanic['age'] = titanic['age'].fillna(titanic['age'].median())
# 결과 확인
print(titanic.isnull().sum())

과제

## day02 과제

seaborn의 **`penguins`**(펭귄 344마리 측정 기록) 데이터로 결측치 처리를 연습합니다. 실제 데이터라 빈칸(`NaN`)이 진짜로 들어 있습니다. 아래 셀을 먼저 실행하세요.

- 수치형 열 : `bill_length_mm`(부리 길이), `bill_depth_mm`(부리 두께), `flipper_length_mm`(날개 길이), `body_mass_g`(몸무게)
- 문자형 열 : `species`(종), `island`(섬), `sex`(성별)

> 💡 결측치 처리는 **"찾기 → 판단(삭제/채우기) → 처리 → 확인"** 의 흐름을 따릅니다.

In [ ]:
import pandas as pd
import seaborn as sns

pg = sns.load_dataset("penguins")
pg.head()

문제 1) 결측치 찾기 — 어디에, 몇 개나 비었나

- `penguins`의 **열별 결측치 개수**를 `isnull().sum()`으로 출력하세요.
- 표 **전체의 결측치 개수**를 한 숫자로 출력하세요. (힌트 : `.isnull().sum().sum()`)

<출력결과>

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64
전체 결측치 개수: 19

In [ ]:
print(pg.isnull().sum())
print(f"전체 결측치 개수 : {pg.isnull().sum().sum()}")

문제 2) 결측치 삭제 — dropna

- `NaN`이 하나라도 있는 **행을 모두 삭제**한 뒤, 남은 행 수를 원래와 비교해 출력하세요. (힌트 : `dropna()`, `len()`)
- 이번엔 **`body_mass_g` 열이 빈 행만** 골라 삭제한 뒤, 남은 행 수를 출력하세요. (힌트 : `dropna(subset=["body_mass_g"])`)
- 특정 열만 기준으로 삭제하면 데이터를 훨씬 아낄 수 있음을 확인하세요.

<출력결과>

전체 삭제 후: 333 / 원래: 344
body_mass_g 기준 삭제 후: 342 / 원래: 344

In [ ]:
pg_dr = pg.dropna()
print(f"전체 삭제 후 : {len(pg_dr)} / 원래 : {len(pg)}")
pg_result = pg.dropna(subset=["body_mass_g"])
print(f"body_mass_g 기준 삭제 후 : {len(pg_result)} / 원래 : {len(pg)}")

문제 3) 결측치 채우기 ① 숫자 — 평균 vs 중앙값

- `body_mass_g`(몸무게)의 **평균**과 **중앙값**을 구해 출력하세요.
- 원래 비어 있던 **3번 행**을, 평균으로 채웠을 때와 중앙값으로 채웠을 때 각각 어떤 값이 되는지 출력해 비교하세요. (힌트 : `fillna(평균)`, `fillna(중앙값)`)

<출력결과>

평균: 4201.75, 중앙값: 4050.0
평균으로 채운 3번 행: 4201.754385964912
중앙값으로 채운 3번 행: 4050.0

In [ ]:

print(f"평균 : {pg["body_mass_g"].mean() : .2f}, 중앙값 : {pg["body_mass_g"].median() : .2f}")
pg_mean = pg["body_mass_g"].fillna(pg["body_mass_g"].mean())
print(f"평균으로 채운 3번 행 : {pg_mean[3]}")
pg_median = pg["body_mass_g"].fillna(pg["body_mass_g"].median())
print(f"중앙값으로 채운 3번 행 : {pg_median[3]}")

문제 4) 결측치 채우기 ② 문자 — 최빈값

- 문자형 열 `sex`(성별)는 평균을 낼 수 없으니 **최빈값(가장 흔한 값)**으로 채웁니다.
- `sex`의 **최빈값**을 구해 출력하세요. (힌트 : `mode()[0]` — `mode()`는 목록을 돌려주므로 맨 앞을 꺼냄)
- 최빈값으로 빈칸을 채운 뒤, `value_counts()`로 성별 개수를 세어 출력하세요. (빈칸 11개가 최빈값 쪽으로 더해집니다)

<출력결과>

sex 최빈값: Male
sex
Male      179
Female    165
Name: count, dtype: int64

In [ ]:

print(f"sex 최빈값 : {pg["sex"].mode()[0]}")
print(pg["sex"].fillna(pg["sex"].mode()[0]).value_counts())

문제 5) 앞/뒤 값으로 채우기 — ffill / bfill

- `body_mass_g`의 빈칸을 **바로 위 값으로 채우기(`ffill`)** 와 **바로 아래 값으로 채우기(`bfill`)** 로 각각 처리하고, 원래 비어 있던 **3번 행**이 어떤 값이 되는지 출력하세요.
- 참고로 2번 행·4번 행의 원래 값도 함께 출력해, ffill은 위(2번), bfill은 아래(4번) 값을 가져옴을 확인하세요.

<출력결과>

ffill 3번 행: 3250.0
bfill 3번 행: 3450.0
(참고) 2번 행: 3250.0 , 4번 행: 3450.0

In [ ]:
print(f"ffll 3번 행 : {pg["body_mass_g"].ffill()[3]}")
print(f"bfill 3번 행 : {pg["body_mass_g"].bfill()[3]}")
print(f"(참고) 2번 행 : {pg["body_mass_g"][2]}, 4번 행 : {pg["body_mass_g"][4]}")

문제 6) 종합 처리 — 판단 기준대로 전체 채우고 확인

- 아래 순서로 `penguins` 전체의 결측치를 없애세요.
    1. **수치형 4개 열**(`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`)의 빈칸을 각 열의 **중앙값**으로 채우기 (반복문 활용)
    2. **문자형 `sex`** 열의 빈칸을 **최빈값**으로 채우기
- 마지막에 **전체 결측치가 0**이 됐는지 확인해 출력하세요.

<출력결과>

처리 후 전체 결측치: 0

In [ ]:
num_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

for i in num_cols : 
    pg[i] = pg[i].fillna(pg[i].median())

pg["sex"] = pg["sex"].fillna(pg["sex"].mode()[0])
print(f"처리 후 전체 결측치 : {pg.isnull().sum().sum()}")